# Resume Milano training in a fresh Kaggle notebook

Enable GPU and Internet. This notebook expects `Milano_Resume_step13410.zip`
in `/kaggle/working/` or attached as an intact ZIP through a private input.
If it is not there, send the new Jupyter connection URL so the verified package
can be transferred. Never paste the connection URL into notebook code.

The package restores exact masks, data identity, latest/best weights, optimizer,
scheduler and RNG/sampler state. Source images are downloaded and checksummed.
The original runtime must match; mismatches stop rather than alter recovery state.
Run the preparation cell, then the training cell. LiteCrack resumes first;
LiteCrack-BA follows. Validation selects checkpoints; test results remain separate.
Recovery exports are written every minute once the restored worker starts training.
Download the ZIP and SHA256 exports, or keep the local backup watcher running.


In [ ]:
from pathlib import Path
import shutil
destination=Path('/kaggle/working')/'Milano_Resume_step13410.zip'
if not destination.exists():
    candidates=list(Path('/kaggle/input').rglob('Milano_Resume_step13410.zip'))
    assert len(candidates)==1, 'Transfer the verified resume ZIP into this session before running'
    shutil.copyfile(candidates[0],destination)
print('Recovery package available:',destination.name)


In [ ]:
import os, sys, json, hashlib, zipfile, base64, io, subprocess, shutil
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch, torchvision, numpy, PIL
assert torch.cuda.is_available(), 'Enable Kaggle GPU before continuing'
ROOT=Path('/kaggle/working/cracklab_milano_v1')
ROOT.mkdir(exist_ok=True)
payload=base64.b64decode('UEsDBBQAAAAIAK+QWV2lp73/MQAAADAAAAAPAAAAc3JjL19faW5pdF9fLnB5U1JSCspPTFFILkpMzlYoTk3PTc0rSSzJzM9TKEotTk0sSs5QKABKJaan6ikpKXEBAFBLAwQUAAAACACwkFldnzSAJHoBAAD9AgAADQAAAHNyYy9hdG9taWMucHltUrluBCEM7ecrLCpQJhSpopWmywekSBlpxSwmIcslYPb4+3Ds7BGFCuzn92w/CCFvSxSzQYgYjNihRZc3oIQ2KOEYdcYEDg8YVwAIB3jSKWv3VWJpMZkTQgZtg48ZfpJ3692nQUVvIYj8bfQMl/B7ea6QjDaoojUMg0QFInurd9umS2vZ2HuIbDNAOTUEU2NoaXaN8iBiaZ3bvdSR9keaPuKCY2936/ft2UuUHJu2jyKeC+PaR6lP9U4Ly3THO0KIqPSpx5ywCE9AOBkhLarGCc82kE6e47m3W89Rl5Z94kr6gI5WYXKcCQORIOWIwt6wDd/mpT3FHlI9xpVZ0jd9TFWBdHY7umLKLM5TdkMVxMVBeh18hNsSlXbCmLvGtao1bd62wHSrY5u/4osz2u3vEBdD2zTb+ikudh6EKR70+n/cNsLOUqyLWSfuiKsmrXxcLjYk2vhG0E4Wl6aX4kf5Vts9nrv5rNr06Qjj6HZeIiVLVs+vhJXN/AJQSwMEFAAAAAgAe2heXX7hXWoFAgAAUgQAABEAAABzcmMvYm91bmRhcmllcy5weXVU226jMBB9z1dYqpTaLdAkVaVVK/qyUj5g9wOQYw9gLb7IHpLy92swpEmUwstobmfOmQGlnfVI0HrRrmpvdTILY4hKobo3ApU1vCM8kP1qJaEmB9sbyf1QIfcNINU8/GPvKxKfB/IHXKcER5Axz0vwgfCjVZIocwSDyjQEZAPRG4FbiEC8AeLAKw0IvpjaOC4lyHJfRGNqn9FtNr0s01ZC+egXmEc2VXjA3htC94XmX5WztttJmtpkr7Hs+SqQX0RYITquHd1E+4ZfZ0OgnW0UhuyCZKJd3pVhVmGvvqIAJ1BNi4l/IL0JyA/dTDkPDoSqlZizRmGsIaAdDmRsFYpLYvvioMwIJnwcqopKeuuG6qSwrdKEy6BpmszZUKXOZQoUBk4VggnW07eC3XKtt3eYOh+3cMtzzvssN6yoO8uRMva5Scr4Htv7wswZD+TvoOOmfSSuuRlytLk1EG0U7ajByEgZwgmebO6ijB353cJhCC0ciedS9bMwBrivEuDVbifXMlj2Fo9mt4BPJROn6zOJnvsFUulQxtPbZa9ps05EblhOFaHXdExgHwSTO0Gf/YuEQoX4CZUTzvp7bPad+ZL6pkustDJ0u5y14F1XJlLr8/yXpfhDaRDWQ7l7Og/wlJq90LPnOXnYZTHkv66+qPRHOLXggdKZftz8muLZzrZFNqGx1X9QSwMEFAAAAAgAsZBZXZr/rMCHBQAASg8AABIAAABzcmMvY2hlY2twb2ludHMucHmtV0uP2zYQvutXsMyhcqFVkqItWhd7KNKktyAIkl4WhkBLlMWaIhWS8q6z8H/vcEi9vN7NFuge1iY5nOc3w8+U0k+mt45XV1KXTJIPx0/alA0xvNQHbo7EKtbZRjtLboVrCHO6FSXptFCOG9J3FXPc5knyhkkJGy0oI2XDyz2KEK3kkbDayzJS6raT3HGiOyda8RU2wXSXEaYq0hl9EBUnlnkhk2hTcfOy7I3VJidv71jpSCuqK95p9K+T7AgfX3phuCWqb4tbbfbc2OtXqE9pUMlr7sCbKmH9ruXKMSe08trAIQXhblm57zsLelomFHENJyUG8r3fs51WVmyFFO6YJ5TSRLSdNo40zDZSbIelAXu6TWqjW9Ix549IPPoAy0Gs70WVBClryjymMh6GVXFrhOMZwY/iH6tVkiQVryErB15MeU1Nr4pKmIz8kJFWV1xmU1IzYn3MvQxfmf9MyIU/TGVGdlJvmSxCKWL6YcW8I6VWtdiBDaZEza0rfOiXtbXcGVHajGy9XFhlRNjCr6/fMWn5ao03Y8hQsw7AAcXr5tvOAxA3jNaOXGMOh4BX40He7mGddsxAXe01wBjc5XcCbOs9LoMsBgJa7kenqU9PywrAtwU40DV5nRGKWYTv+JnjLTBYunQ1hUvHHIPg+P0x4bEKIDx+XwgTUU8nkCoArSPvteKEQ7bw20IfG5Sxy5rYc9Rg1UFLrD6dlR9252CYmZ6jwnuwRAkNMIGDiJfp5gI4Pr0LIMF5QI0/ifiZ7s6ABOdzWE0yRnmz97Q7ugZrGZox33GHzkFFCEWkwZnq8um4GM4foJkiBEEeP1EWzDwhX/YVG8X9YnmngJESSjSTgMZgByYk20pQOlXqFPSf8P8L8lmJLz2H3mh750VJLSTMOxhtlpsDjFIJY9LPql7CjOYHrryds/lMajCUo0bFWt8NNfUVvrqflXv96rfqdHXvp1Tu//2UrvKG392sf92c8s5RvO6YgdBAAfbmS1SHB/P5lQapjEjWbisGLWg4a4f8+FGWRuCEk1Xo1MFp6FXqo/QFA/VQP9uwH3/+BdZx8OZhI9rJQUdVbI8QaLpCnyuxA7BcKNXzwR/yP83hNAZMpU+oy/0ezQafQwCQ+Dju1qPhCxq2j9033PVGxRzHwS81xPa/D/67jpfw6Bdn3TjuD2P/thFlcx1jpsvxja/TbP3NcQ7pQX04nuC1vR/0ZiEn9DSlzTAB/fA3kz1/a4w2KQ1XkV9sOQk3iTY4Fujq0fcCwGG1BMSdY8y7n/vs2nSoTE3v0copVGcVgOX4HUBptYT/cCXquwmA3Zyb8y0fMBreKfJd6JynAn0zsSdPJgiHSnbQwxARgYigQNoc6aj/P7aE92B0OvbV5pnuIAxt3wIPsy0DYhW9eEHe9VJOOHz58f1fE4McOVonyr3ka4R0IIZH3RviAgGdmGa+eLfD0PB3xrHSsq7wdNWTuWtadj0g6JaLXeNs4fVGsjFkCDXdnL/6G5+J1x5B8Xz5UuHx5T55ZraCPW8AZjCzAJtBy3n+Jh/jO7o0HjaftIoSvcGMQJmY2vHqd1ICCiCHjCh+i/ARCjLtk1/ji9EKeDyDCzW4uefHDDk6vEIKO/QmnZGY2XSBNzUd6MgwaVYzFEFAYcLfgM6NZyT+ZUPspTMDcXu9mFEPoqvpxwFK090hg/D0g4lTjCJwNxyZM3I01BcJ3iZITuTtEemJ6W2mIl2iapP3E8d7ROeUy7nOc9I2V8ie1MZmqiKvsQPtiUKeHm1uBnYUZScaZEcatJQPhCmKx2d7wYKW4oEvbc7RHE+RHW0uhwjCfvNRXvQNbNDwu/DN5z//uDBwwi9OaArPm1CGq4MwWvmfghEyU4Bo3T7gbZdiWTzXKJD8C1BLAwQUAAAACAAYt1ld3eRBTqADAACjCAAADgAAAHNyYy9kYXRhc2V0LnB5lVVdb5w4FH2fX+HOC1ilNJ0qq2paKm272VWkblulq30ZIeTAZcaKsaltkrBR/vte2zDDhJHa8gDGvj7345xrL5fLv5nkNRj7QknRk4pZRlhZgjFvyfcOdE80lEpXhpRMSmUJSAua2B0QqxmXXG6JaQW36XK5XPCmVdqSHTM7wa8XtVYNaZl1P2RY+4q/o51mslLN+Ce7pu0JM0S2YefXy0/jrsuGbSEJny+tGbdYpcvdYrGooCa3oHndFzUXYOIh6EQrZel6QfBxw8x5j/1kqsEocQsx9au10mhxR7gcEw67xrWag6jcahxxF0SURA0zNxE9mLnHJZt5By8RbeN35U99jQ+viauo25NyU2gQzPJbKKwKIa6xQNwA+ZeJDi60VjqO/kCCDIRNBEzJWjA+tWiGPbCQmh1bnf8WezcaWFVc9xZLRGm6g/uKb5H8mD7L9vE+j4qwJcqPk/NVfBpRvRxDKndQ3piuwQGTW6jW5MFhhnoVvIryR/LgPTwuKZJWCmYM+ahZeTNABHeOzKJAadmiiA2IOmkGjXo6Ey+3xPD/IMMgE9ZtGxRl9icTBiZ0YAWGJYIyCyJ9lkVGdbrEGjvxRqcq/MX1wdSKNKwfoSZVdpGlmKDJNjrIh3jbEOomGmQU5S4SvYl8AFGeZX6QT+N0KtjDnYrpomltH1JYk+i5H8wiOZL32zDrq+Rew/9YrOE7qbcAOZTb6Q5spyXBuXgfF50Yb8FyC83ID5cV3E9Kj+bZft/Grx7yveOoXN/IqWoPDoaOGborp+4gCCysiZ/Lwl9aKomtbuPo6q8PEzp+AOub9Qj1SNqa3WWyTZlhWrM+DiazjkKrVFa8ybLX884YeHQoQsRouknTNDlDwsfxK/SPOvmB1Sqnc/CTrRd9VEJpqIhLDjn73nE8aAjcozxKbnG6bfF8fnIyjOkefJ86lLAbYwyykxxvARcnTa0S3B0V9F32cJaszs8fT0n1s5LXXDK8OML5eATuprJAkjvhQ7FdVZmxfQveI5f2DT1sC9z7tztGUcpxvJd2sh/RJMBegWEN5i+36YfLT5efL36/OmB59+71i0gO5eLbP3Tas9N2Ws+F4i62NHxi+i49H0ScHCqA11jacF80vzT4nUw72xMa/DnoWvB2Buwnj2GHVo/9VepZKfxF7LgI/AT0yhHkeqQWitnXK9QDxmBaZSBeJWfJK/oSJUGTmdZmwAdkH8n7Mzqhf0TfoI4gRx38D1BLAwQUAAAACACjaF5d9l+uVvkDAABjCgAAEAAAAHNyYy9saXRlY3JhY2sucHmVVsFu4zYQvfsrCO9hKZQRame7WHihyy6QRdE0hyx6MgyBFsc2EYpUScp28vUdUpZM2Umb+pAQ4sybx5k3Q06n00fYSqNvjFbPpDJ1wytPaiNAfSXakLVpteD2meyAC2IsaSx4y6UGQeDowWquiOCe59PpdCLrxlhPvLHVbrKxpu6W5PRd6+RjrnX/fdPqyiMJhOKO3E0mEwEbYqFRvIJyzX2108bWFGm1CrLFhOBvg2Q0r4FVO6kEkZoo6fzJJg87ooxbFjTNTk7hJzdEOqmd57oCGk2Y1vm3EOYBw8xFYhx+Djz33p6gWQyKDj+saZvgQGcdh1y3dbkB7lsLLssGDFAOFq8cJzpleNpKcefIT2i45WsF35SpnihG+DM9b0hJWUotfVlSB2rDpMZ6Sb1lpvVbg4uEt2sbsDTLB48zneCbK/4M1hUY5Cf83YL2kis6OjVufTd6Pxd0iDMsblnDhcBVMWPbkAZXDHtryV1xx/HMGfsvwJ44m43cLpI7HC9sPML9X/RN5AFwWLxCddh7B9VrwP9LNRuqh4o9cCu64h2zIAmUik4LQo9nPdxLD98tf5cUDlL4XVm3yhf5b6xv2pJjPDhRfZc2zu25xx4xOo+TwPWNWpu1VKDBl/vb0tVcqcHzAzmA3O68Kx6MhjAm9qgqhwPE74gwB60MF45wLUirsRcQDkfI7zXfwgN4nCWNcdg3+VimgEoRYIvrwDQNl57/vMzyvhkH0OsmTMNc9IjUQlbgCvorFv2WfWGz+dmi2nGtMTXFMkVYylWOGij73TilZBhOKeJqQBEQ3cpIGiPN5l/Y50/sds5mn5NYrcUh5osedXkzW42pnnCKQSn3YRYmdUUa7kk2XaICnxfZ0FAj60DQAXiByBkbs7qahud4OW8a0IJejK4T31/OEbPs63CK+GFM38IGb5TidRh2+n9RHUxz0/ri3Kq9+YzNLkz7fohaSe+BcaOMz/mB/AHQEL8DVE24IknoFhyT8oWHyypK2bVr181P4njdKLDk8eEHkSJM1IqrC0RvErz+ytXeGhUrFPYQBv8AiHzke5BYtu7etBjY1Dk6PJVWb6mAfZTpcnVRqevz/0uyXhtQA5qxcivxei6dfIHimLsdb2B5M18kMsRaBw4jyaHe4cjiaAuaA7wgwXIP464bsz4W0T5MwvRzuLUD2mUvLbrIvRKPY82vg5BYsOg1n8qXDQ0QMTr1X9G5w1D40GmMCsyPLOYgOCRpYGFOFh81oIyc/5hdQEQatCtfxT1dHiOnFROyLmbJU+GNaKP8vxmrn3ZF0lNpQjrZFUn30OG1khjFO4l2xmykn7N5Fuox2sM3Fb4XPYnTP7x4TuEm/wBQSwMEFAAAAAgAfWheXXJnxkDzAAAAwwEAABQAAABzcmMvbW9kZWxfZmFjdG9yeS5weW2QP2vEMAzFd38K4cUJHNm6FG4q3UqX0qmU4NhKImrLQXHon09fJ3fNleM0Wc/S04+ntX78mgI5yhCTxwAzBnSZEsM8WkEP3TdkscTEA1j2MEnqKZSu0VqrXlKEWVyzbc9AcUqS4QWHiJztavT6jFkp5bGHbqHg2220col7Gup7BaWoh1P/Zqy4kXJhWATNOxBDZcIqiHUf5nB5t5015/W1dpJ94A/mqQgPq7DPChZ3vnycYZoBc2WQXQGU9pN8Htu4hGwOzV19uA14PF4RqX8HrmOobnvs1p2dsXWjZS5Z7voJi5NEG+hnMys5cGI0da1+AVBLAwQUAAAACADyulldZkBMbEMEAADQDAAADQAAAHNyYy9tb2RlbHMucHmtVk1v2zgQvftXEN6DpFhWagftwYUO3Y8UC7Q5pPBeAsOgJdriRia1JBUr+fU7Q31YlB2vga4QILLImfdm3nCG4/F4OX1ghlCREkqSnFGVv5KcbljOUlLQgqkpF7rgCn4qlpRKMWFuqTHwj0tBNlSznAsWjcfjEd8XUhlipEqy0VbJff1Kmu9C9D5GQrTft6VI0BnNCdXkfjQapWxLhFR7mvM3ikvrnL4y5ScZFYLlOnQWg8WIwKOYKZUAlOirkmXxAFv8WdiaBIQPfMaxt8ONHoFlhnZ/phiVefUD4JDkVGvymxQvv+YyefZh/btMy5w1cMhxveaCm/XahyRsQ8JFIvdc7EIiS7OT+OYiekIK5jUO8NElZNgPos5RcFwCl5ENW8eA/YP9UyI7miMTpDVP/Q6ww7sLC5qm8BbPgvBcCs8zg70iemTfln4Qdgzef44MOnf/I4Ogy+9WqgNVaZ3eKli0Gvdy41dHsR7bAn1kmqdlnar/EO00hT8pWaHk3ywx8SWVZgObEjxZlWuy37g2/tO1ScYskTUUH1FU7Jg/D1YD9xjRqfvrpXkX4ZxGHbRqRIj7aUG52g1Z3G7pPiEQJsMSQMA3XvjHFIXHcHpA1pBXLI0z51sW30eK5aWPBj468O22IAhc22F0i3dNJ1nfuKnGNopJ1lXil7ZBfqWGXVGF+pkX66677cCo+3Vd6WU8heYV72kFPc/xdns7H5QDLvfK08WuHZ1UKFLq2TgM37PRiVR9o3Zbu/GkepCJDd6povrEw9JNPTg03+0lr01qDL8Rq4vOxhRMOuaWb4DStQL9YLs9KGTrewkT8AqNKIBzAzVcQlReKZjxQhx/XR7i2aefaB4wnvoIMKsM1qTfIHlqTksPWyDlMK3+onnJ/lBKKt9bimchD8Ix9xzHDqujZ8subKbgRdeOg57vDY7GuBuSwyDiJk/1hD1pz52bA09NpuMnJ50385sbbg8nPx7Oj8HKLTIGzTVtZmS/vVlm/h0y4nH8oWZQ4zzx6WwVHs41OR4eEIuJcs8UFk5tcdJRU3YJtYWZzFaT9h0Au7dzwMcQ78LpDP5OMLtL1xDUbTangEDjAsgZzWyp1Rl7gBJxacCMKMr+eGtgPqwunOz+XMCzCVqvnKbPw0ZIN/2OxIOWj8ouSAWdGrreupAyBzJVOHebexU35jB5PtfYES0KJtL+KKopNKqeodDqPaBgO6n1+TSdT/nKWfyFPMB1mmljB8QbjFVCX6BxwaVy+fsXsuF4b6ZquqHJM2YKDplImWEK7gpc4/QbuDMZI9oonqDHRL4w9drdvz8TuGMUNDHTPRDNCaug0XBscRrUBTtWUAwnGiTnPuICIAuZY6xVCESZjSjSGdz+IazFKkSX0MvqaLxgqIJbnYRr22GwchZNhpwNUJq2P4fVUKkmy37d5xMKtV3ZobAKU76Hq87J6O1VJMr5L1BLAwQUAAAACADzulldUZ30au0BAADEBAAAEwAAAHNyYy9zZWdtZW50YXRpb24ucHmVk8+O2jAQxu95CiuH4kAIhao9rJQeKnXfoL1GJpmEEf4TjZ1SVHWfvY4dWGALaiMOZDzfb+bzTNI0/YJa0JFJY+1KgSOsLTug2zH42Uus0bEeaIlKdMDEDyDRoe6Y0A0D1bvjUgm7ZzRIsEWapgmq3pBjzlC9S1oyKv4ttGbTUTvo2qHRQjJh2XOSJA20rMEaqrEJLk2HzuZOUAcue0qYf3oyW7FFie5YxvPCYqcMNjwLCagdkIUALvlF+nziFHZQnK/zTf4hi5IGtFHevO/vSrG4qyBwA2nG10u+mV9WXKxh+Slb8QtkDGWFAqF9j9EkQeez/2Iz36Nuytl4CbPJMrYsBKfo06n6vYu6EW191rXwudiGUVc1eXUF2pHpj9U46yqiboiLh6VIoAX2XcgBvhIZ4rNveq/NQU8uw0bNTs7D/lTTft0ZMfh6Dpo3A/5cFh9DiqPB7cqoOgcbVKCtL2jLaVgxtx+HOiHZu6iNM31VZEUrjXDTDrU3kpd/0fhte/nPOuFa/aqUm7nrF63/6RAf9Li8V6FpdL+mHYif0mEHBPxMKd/n3rdHrU6hopZC9ZXfRL7OsjyQTs8MzXANimUDxTPC20OAd1vjaOsaE/oeMbwdaWfLIzQeZg+xniqkfMt8bS2+3lB+J38AUEsDBBQAAAAIAHe5aF2X6B2X9Q4AACosAAAZAAAAc2NyaXB0cy90cmFpbl9iYXNlbGluZS5web1abW/cNhL+7l+hS3GQZMuKHbTF3ToKLpekRXBpEqRpD7jFQqBX3F3VWkknSo63hv/7PTMkRUm7dpJ7WyCxSA2Hw3mfoR49evRStrLZ5mWu2nzpqbxcF/I0k9f5UnqXQskiL6XXNgIQ5TryGqm6rbgspCdWWOhVdZtv89/xpFpZq/jRo0dH+baumtYTzboWjZJ2vBFqU+SXdqj/YCLeylZkohX2zW+qKu1zpY5WTbX1atHSYrPKe4+hBWlEmVVbO1L5uhRFP9p0be5GO2UfQbU8OqpULMvrvKnKWMk2kyvRFW3gv/jlr2+e/5z+/d2Hv/38/vmLV+mLd29/eP2jH/mzb8/+/P3sT354BFwxERXnpZJNG5xFqm0CoitI01VeyDQNY3CrKq5lEAK0kWWr5ueLMLQ0lN223nlCeWXdk1U1y40+sWqWsWirLaRiXn5q8lamzJ0eYrmRy6u6yoHbgilxLVM3HxWVyAZjt5Z4jmPbdS8asbx6qeeia9nkqx0fRLkV2yqTRboSS9C5s+suu7zIUn7lIJVcb3Fg0eZVaQEz6FRaVEpF+VasZQq5N/lSRY1cA4rfOASXVVdmosllfy4zs9Mo+tHq/OjoCKLzVNU12IDULAhnRx5+TVW1yRfIhIFLsZUqmfvY/TGfBgLeQeY0LsD5JfHHTTkC3dyIP27aMNpNDLnjZgeyNJPLJq9b9ZjNL7XWSO80xY1su6a0hhWrjXjy3ffBpe/HvwFLENDxH5d0ZGjA5a6VKgi9VdV4EEmpzxuG8UbeZPlaqjYIwcm/sArGZZWuG5Fhilgrr0XRiVYGfMDInCfSbiK6FO1ykyp4AcN2hoppURBeQKdbUajk9kruZmcx749HoiDwSSVw0Lzq8H/dyGWuwBI841EUBR4GcvbDO0ZfyyZlDUrmiwuPtCGFU0rOLjxgAMZWZukKwiLuJmcxKGi6djOcYjREiGoFuaKSnMhawogLWQbmeOH+weiXl0S0Sgp4zEAvYywRnKh+OnHrRvjw67Eosa0LUjfzcp4vmKCciDFbLHrom0RLBeiXV8Fczc80tCJog2oRxm0VaJG4fXbTledfurKo1nmrEpZlcOPm85UHIZVAWC5loKGitgOicGYX6T+gsl+1L5mTZAXP1AYGxbPkLIz1TIiIIMogDI+Je4YZjoCxNC2a3cOLrJZY8N4X2RPs7ltJqg9BjVyWWzQBm4/0dZEMRgfWkCB+i7CjvCFxSAQE2ZCZWSKc1mlzX1ZNltz6mpY882fWszTVJzVnRIu5e724G603lgffTv6JdtREx3BuWxVMdusPZTimV81/W/QmPQeyxUnCQBeGOp7TUyNsvc3Goq5lmQUaPBz6sVufBOLPrKwej4zx+Pj2anY9mtMniq7pKJoke5S7aLS7P1C/CuGmIbH0KuTP9rVzvPcYmVHAQ4jGujlG4uP9cOIu6pliIhj7+WBZlat8HW1Fma/glVPKMtjlpuTPo6Yr0yxvIs7EZPIDTi2Re1R1yhlZynlY8rYqZbTKkQqlxnmT5/vYdNaVAY2OjAaflgSMu6xaneUhxSszgovlDXwdxQ6aEOWO1hCnG6yDxTm9Af1Ker+S9F81TdUE/oeuRNxvpMkYFKOX27rdXXgdYE9PzV6QpPBK+Yk29ENLYry9oj1MpGb6I6Ymra70aXTAGfJKH2s0NYZKKIWKKS1SYzAdKFt5Ax/U88NCxGuJ1BBur+2UH/I5OIT98OHdP169Td+/fvPuI+KVGf70+s3zt+/SXxG0HmLPq5takt55yHt+l2WfaMNfVm21rAr/M3Qkyf6OLCatRRrYIftD4m/zQiC6X5+ndXeJILYhnamR4MDyHqD0J17WUwEV+WcHuSKOIPOUN0CwzNt76e/ZTLlKMklYDsjAJCvj5IQQ6WMlt8fHxkx8k/ZpXP5slAVGPke/FNms0vbJwTC1E2k6Me1ll8HMupLKAwttYGN6N/UEBHDNOYvbY7+wsSiC4QLoEKSqTzH3BeYpx4QjlP4i5jxCfcqhyH6fe2IFklLpkW3fWbUYypnd50A1TXZ1CX7wIwL0vmB/Ka/K6lPp6cVjtAhnfSrjL56ek5XaNy5bhYeRCHHseKZQOhbZ9d8/pGGvSziqPPN4y0EyDNuArslyuXvM2DzGpinVSYvJcfQgsFvrob/oz6Qn4nZXS5gNiVObCvFLY6C5OFepuBZQd1S5Qc+yD1opDK0vfnn53OvKHm6fcdmwrvYX7uB6J/i+dASSimJdocDbIHw5x+YWXEIBEDcV0ViW8SX4sdmK5koHAO0vuRBG6SWzngk08Clkl3X84Hu9C0yxQ8g4BPIAF2dD9g1Q4EzFQTTfeL9yfekBgMsX+JGqpI6DpLDauxGQJRqotkdWxNGDVI0CR8axv6vxAgkMIx2WrL1Pmfs60YBiuhiqibCbJMPCt184iLjWxfACPzqk2pETGSuxDrdfitgtOYw9dFVVMqi2DWv3sveJV9iKG3jWBtUe1I38A+k8Ym5Qx5RwFqYmrMln6MrNQcMFP7MUTfAsHrLlF9W2Rg50yvi8fpV32WUkTnmzhDZAHzTBfRPJGDKP4+eZ2Ab7BEVFk1iSCikaEmFKWbPlk4LryLpigq1AlLMv4g/4u5RvPrwr3xdYKbqgJyGiHSlKQhaIRznsTCbfRbqmT+JzvYesq+UmIY+XLLtG4c0lohbPKipEoXHYnxz1BXQaKcfpOerQDSwdaRDKVkbCeS7nrVylwkBQm1oJ6rTI8ZgCvkwmvRxKxB4PPLHyI60luk53fHXHc9zpn8ZBTaHwNrQj1dLpSTqO36ORAzKx2Whlj5S5EhGvImYFn2Tu8zQsxwzXRXVJbgNgbpLgTc3lu1pSl62NRWSGKQ8HYCyDSIsnGojHwM99BqDN7ISGHc64ZQPEVow9lJnwbTdCy7R/7eYIgsXcv6ORQU2x3cnbNfuMkLVF04R1EuG9wH2uqMHtUC+YApsOKDWkDPytX+/aDeUy1OU0yYvJpR7MoWzMnVEv1LikyF/XnV3FEQIeINUvU+pBWcD7QnSf8wz2Gfih2Ze7sjvjIFAs1eT4b33OLKBzKKU5jt6ZrGLlmTcpAQfHKfIAu2w+WLXgIkQ7MbjlDVwrxKqS2zvXYsrXnI3pxnT88+sfX7/9GLnRx1cffhrUCEM0c0AtEgOq/xCaaEhb2FOsSlGrTdUGSGDY0PhEA9TfeC8rTnbAGLDEg/Mscurcm4UKYRyZ+hUyLDhvGXtvpLiWXNhD1tz01/1ghxA+B46RI8oKqVBfCBQ7G8qFV+RXEuOqa0+r1Smj1joYOxeBDGgL682SkTCP6xiUkmJyIPyMdI+/HfaodOc/znJ1lXYKsZSUPYyRS0rvqdvwBEXD8fnZk2+Pj59Mui0H0r7XpepWKxwQJHmDDJV20Uy7IBbwrQT8dwGWcg1tStyS2iAKzoKWk8sAWf6wJzhq3P/PnTvHMO2dB96Xo9pk1dDDwmq045xpB2u95sx42qHPnLnnMcbJr/egM/MQDZ3mzD1H2mHO6P9Jj2cUfaKHApZp4iVmr/np+YI0xgy1v7mdYB9EooSeI2tk5q/1LCjboMWkMvG2gq1VZb4MbL65G7j3DSxJh8anNpuxgYZ1xnGuf2/TkWHqZbSdRUEJMkl2v4+n0xHtgqkKgBpuA+pF2Rw4pCSSO9ooBUxGc3bxQIZif9oOdeMqHL3RJ9S4ntJeTMSBJqNtqPP7uV5glOnkUBm62MPQN9PtcR7uptvfv99Vt79/v7veS8babox/5s4F1QKYnpYQZTIuBe3P9Cupxtjrz/dsHfXp3Qrbqz9ojyTwZHAfN1gH7kS7vjzhbsEiPBkWGn2z+5PM15vWj+In4fHo1m6I7nwxbIXb3zgJ+kLS9uk6xA1X5udqBTVp6fZCqXsq/LeEkIBcT4ugLyBZRfnuoE85pZJr9U+iyYKDdAxZRvKmYJIui7yGxJutf49kzLVcGVNYUzHD02pelB6qkyxLDuyxiCSdMs1XpGX6nPeomlNQigsHTmQsNRlenlzw1wAnyflB/miPYm9imGOZhAVtprc32uOMUe8bP5Kg5GBuRn2oYNoZtz1ocpS6Esbss2QKdlBy9OaPn+t/JcmZxzeLVT1zOdk9+ABzUNrTHF23e/v0XJc6M5+PjcpLiRVSLD8yRdVMB3Wfi6nZfjz/sp8vCxBP2JEAlhnEhgiJgTKt1EmMOzXB727/qPTTdzzj2GHu/9Nkcr/s+iH2ivlQHBjv44p71lOD+7BDyLd1g5Q2S3oobo8unlEkn4ZWCzzThfxkyYU3LPxHa9mVDV5PzcHkG/ZC7HYkvJPzkfh8/QnA4H6Mbegx2cfDsvUHzaWZ5fikcTJzRs4eJKV7rZriIfiHengi0rqh7JQvUbJuW6tgkEiF0arokHHtOxN9rOR83Bz5mlzDFThGJOMNvvE+6Epl16dysHXRoZKFNYED13xLWRdUgBTVWsHvtxukTS3qLU9Lwcs6KmBcU9H+pgZphact0ozuL8jvsV8bWtJlRblCK7/YhHVrxMlWN/hnnJr+54YLtZ/eHY791P+hD8US/+/0oiLkoih1mGeTmM3HtOV6P0n+m67EKI0bhmvHDntlFs1tE3eDyh01LiSoe9Yp4hD1hA7Ec3MD/Nn+MO/xQEd4+LOmzVBq6lEnX+x83p32vJkzEQvr9/aAJpoeMPiJn7oLf1b60BAWPmhVjr+9qTilgr0M9J1lCV5vUFwT6/1Z/B19NsRVtz9j2o1GU7e5br1X/Ie+RxPK4/znQGuEanDqV7e5KArUgjdy2dH97OQ7R4/8jG712/7ITrdo43t9ADVIukb2J2MSdKuMH8Op7ReCezzWAQzskxPWo15GxaCwNC2nyPSQSIOHLaX+Ww9vv61kYOgrMDiAlJtzaZokfgqbQ/RJzQVxndhPO+Pnzbqj/sx7GlHLT3+yRoVJmmbVMk010XUssiwVBjrwT0+1CvqRuUfO9N0+N/7e9zf2+8uslXz1QrKAU7Kqr14J4Z1mefP167jr40fmqzNK1hq6ROpsz2d/CeVzp5yFnupsUu8C5dcrRFLHzHhapfq+AhXfg28aRGx81vBzhohv5rSHEbHzMSK2H5SIWFOMh/1s+F9QSwMEFAAAAAgAd7loXVuE22dGCwAAlBsAABkAAABzY3JpcHRzL3ByZXBhcmVfbWlsYW5vLnB5nVlrb9w2Fv3uX8Gm2IqyZc3DSZCOowDZNrsIkCZBU/RDZ70CR+J4GOtVURrb9fq/77kkpZFkJ9uuYXhGIu+D93ku/eTJk4+1rEQtmSiY0mUmGpkGLNnJ5Eq3eS5T9pPKRFGypKxuz1kh97JmImvwt2o3mdI7fNuqTOrwyZMnRyqvyrphor4EUy2PtnWZgzTLZNKostDMbfihbAvwCFgqt6LNmlQlTUe8E3qXqU33+FmXheVTiYYWOh4f8dht0ru2UdlR91i0eXXLhGZFZUk/vn3Xkb3NxaUM7MePtbh2Xz9U+ujoCOqAl+Akyl8dMfzUsmnrotMqxOry2XNOwu2usJYijTe3jdTc98OdvEnVpdQN9x2/oqxzkak/JM+FvnJcv2VvbqpMJaphWZurQhSJZFWJN7CyyKqdYHmrcZISBlEp+ecWD8XpJhPJFZ6wIMikoWGntoyYh3mJrVHEvJ//+ffXnhVFP4JFsEUotKhrcWsV6RdBTGvFLediHYZhwM4u2Eu2fPbMZ991S25lhaVvIjYPmLhROlr6/kGIsZZQWrJfRdbKN3Vd1tx73yvd1KLQFGxFY7RllbqRmYaFf28VQrCWeyWvPasYPPNQ5zApCwRgw713nu8P3WMcyTlRvYrYYvnCB2FzW0kOFq0qmhc+OzZHcl6pbNxzXbZ1gngo26ZqG3ea8UvoYdxt35LDkSh7yf3Avnekh/edS+xCKGGpBsFxsNRDK32wkuzWc9ZiWSDbrhmOq+HmgBTWst6r4hJfVVm7E5gg8KYilY5riWRWexk3Zac5A5X9Ot0wOv3jGn4yhIi8tLNdaoqCjdONZNqqI+FnKbVTqRKq1jDg+sI8bkkDxH3DVMG4h4hQhRcwby8y+kAONd5Ai8u6bKsBecfiSkELw0FR7mqipfjQ3iQcTWUCvUbmy5RXhroypNYQM6PNjBj6oUJJSlXNfbJkRTYieu77I565qCryQsTuqlA3Ml+xAV8j8X5EAGaZLLhZ8Sl76Mlxmej7uOl/bKlUkGlJXGfasZFC8JNF2vMdZrfdsZ5fhFfyFoFIOrh3i/4dDkDFpt/7v/L6J6W1CUby8Ip57MR69iDZrCD6G9KLm8WAFSJHYvVS1vR8ERzUsS98Y1D6SjZ13uuJ+tQ3dSMl/7Y5J2cIlF18xBrl1rCIA/pVVcBy4x4bj4cosCt+nz+2kYSp0ldxqxFbLjVCW7f8cIvgRnHsZZ+wxXz59Pj47Gu587bQ7XaLak+lj3izHZpGXZa586UTkl+Z8HOnS8o61QGjU+FjC2PLGqlfNDYjAnZ3H3SZkWJ3akwx6Kgc/dn5o0+8zgMTkxy0b0SW3ZJJSWyoZeMYdg50rXuYFc5GLpe6nPTdcazldPRL3UKuqXBxeWUev8jCJfNf4WCkxtSSofyjCs1UFdLhexKS8jiFlT/jtP3EC6vicpB01woUBjSEJXIOMeQT1lB5MHyb27fEaZxKKg+zUqTcP7c92z5MKwZ2mSBGqppd9PBnagXSEmDjskAspCqXBTUPl550mLGc+nKDo0NU31eBGyblxXTWDRBKTVExATOjnRb3YNMEK+mm5pBkjuCHsqBI5f4JvWrKR4HTyBYpfK8aE5LknBNv5p2M3DiI/7VlcdFVw454zPJb9hqnhaeMpsC96LU6QWIXVNB2sq0RYSoJHNytkKjorCgOaBDImFQSb5xDhiO2za7NN2PQQmccYhbCCGQ8/iJgL/wuXn6WWuQ4W3EZvnv9/offPnyatJyNMjmP1A8Jzjq0CTkVcBUtcisb0Md8CXMpCjJqb180yI26nLh2WFI6i/UmC7p6Qfwn+li1SRV7TBseqLxibwPDZNWYxtSVsK1S9DFuak60QDLnVXMbm3SLoAanFmS5GdTpQ3GULkmlv0hjCxjNRifz1RxSUfyx8UFy9D8DMsfbhGJABpU1hhwFnUbMCUiC9Xf0+ZJQ45+QkuwEzDlWEuTfRL11DIeDVVyvIRBlcAaV5EMVO+xznaDz0J2tZ7FKPeR1bsYzpETvty9q6BmHgujOAa+VZ0FQ7HCYgWH9Szyo1MLLwCKzfmlXZikKpne/NiwvviKznxJjJ91FlT2EtzocOHwEkCKR4qrU6oZ/xfKmWnurPu7+f0ZWGVu2oCkGwYE/AiNntHoI9a8wRQzEDykr8HNFKx6LtSXsfoTgCDG6MPD/tnw2j6L5uBmYJOZb72MHze+GJPcz82iyzr8HWt5mrd4NGqi8EQlV7vXeIIU94QKnXbin7qItKCYudzeh8SFHIfaBxgzFDVHs7/1XC4tHtKLArKMB7lcY0Cz2GBYWopOY1yWNDnxYkQbQmMjLZkcXBuYjdhzsg6mOBnofiNfqZLGaYFiCd2b2QE8dsHnYVYHQGhrII06s/32Q4of4GyeEgCZdyvHvKF9GZw/ZDuxyyGXhrYxRvI23skf0hG1OMTWnuONIcWG/ucjIRaG2CJToziOo1mqk5z9+/vDbm/fxT2/RRj7Evy6QuDrZyVzEboz0VotppHpob02ZlJT5tpjE+0U8SVtNnEzuY9euaSq9ms3+kEWZlmFZX85coM2W8/nT+dOzp94DKR1wsAVl5X1UWVZes3eD649XEeZ2dvoK9fbcOuia0M38HKiDmcYJBz/knImNzGKNMkrKJTVaorZjKgyZO8SNmVRjrj7cWdljnT/s6ayldq22SqYPZblzeiv35d5BdbpZco4gJTZZmVwhsU1axSX4ZaLyKEBsoqE/SDK8zVWaXp2VVxbsj8T2ZSKpS62tP+K0GwlBY3jCPza00ALiBMen2k2r7vWE52CzvXWC1s+fniK8mRhioy7oGCL6nB34srLIABHgGNE2gAGASzBDXppBfiwqgyh7VwVt1t770th8loiqaWvJctkIsBTsYHXvU6/dUGQhUdfcJdFX+jBEQkYNVH0K4ACxBkDrc3N9Rq/ZVsD+p9QOqEsw+ECQHIM9SCGh2/oR749l/EI9EwF5SpZge6VbcEUk6UommFbPGbVMVsl6S5FPJoR9QKIRYDQgQMDFfV8cCQDY4mjKLZWzNfe6HA/pEtQLukc/4N7g9icWbaq6PTYW/UHpo/4UufGmEzNaDK9haTRLTOqcmIRpm1eaG0UCSo2iiZb+ifcvsDfoHaeOvLbZnr7w/AknjEaxmXVvzO1oaL9jhHLdzR8K629aTzwY9KRTzkiyjA0wida1sVFNVnGJR7lUrx+gi4soctjGtaGdlE1k4Wohr+18E/Dvn8+DxfJ48WKOLny9g0IeprEUnTrq74VD+sMNg8MI/dnY42bctIDDM/K7SCSfB9QijQr+6QJCgpRuIQkNDq9K6/I6MpvWht/hdms6Wzq/Yf/aIaaLx4bN4TaDhuyufAISdqj00YNxD8XWVKhbZ6ikzE1uSG6fNxm1KkMcTE1p3ho4HXiUM34QPvft2yCfTBuwXxJQ9o7N5zhPtQqcUtMb5t6tGH50p+OHShN5A4tykhDws8WzYEHAPeDJ8dlyHnwmd58slwMATg4PTSTyZ279zA8OtiaIfYHY/I+F5Wx2GH9Te4M960znBYjeLPLMTbcLXqulmYmcf2xsIn1iW8bCzxUa2u+oHCh10fdLd2dqsNwgE++Oj21af7HIU9C5Jf/+kLPBFOd1Hehrl1VvTIsyzebU4iXXwFasU5+5Btfd1bs7eKPj0dERpMQxZXIcIx/jmMpeHLt/R1RR99+h8HV92VKD/khPNU8lhnBVUUmL4jgtkzh2BglFmsbC7ebe6anDIoHJLbqED9wtlXngHjWUGd081oXIZg7WdHggXs6Xz2kVUKGhQ3xJivXaX5XiujpQVMdZRDSp44zEXdPFT/ffBxG6fzWI0A0qR/8FUEsDBBQAAAAIAA+7aF3WDUfmKwEAAP4BAAAdAAAAY29uZmlncy9taWxhbm9fbGl0ZWNyYWNrLmpzb25VUctOAzEMvPcrqj2jare0CPErCEVuYm2s5iXH2UIR/04e9MAxM/Z4ZvK92+8nYG1JUEthnN72k2sPBn2dnhqNQUeDrG5kxCpfnNSh+XDuZIjswdEdhGJoyyvHksYiBRL6RzIEE70KUSVGYaCARt2QVit57GREUydPx6HgYUWV6d58Hc8vHbyAaPsAX4fFFLXN9bnMcwccAgcKq2IQ7HbneV46laqbGqmhy8igLeprihRE4Yb8pbJgamrPQ8zgRn1+0sXA8OlibhPTRaMyje3oymCquCjtKKnWTbtyeMgIsq+lZCFdceGCw/0GrvSOVE61/Cb8XolWRyxcL9joTCwyVfCjr3j4VAkYfNMcwc/neTmNhBwl6uiaP08Oat3bolK5OMq2fuTfkd3PL1BLAwQUAAAACAAPu2hdVJ2x6zoBAAAbAgAAIAAAAGNvbmZpZ3MvbWlsYW5vX2xpdGVjcmFja19iYS5qc29uVVHLbsMwDLvvK4qchyLpmmHYrwyDodhCLNSxDVlOtw779/nRHnY0RVEk/fN0OAzA2pKglsw4vB8GVx8M+qIWGJ4rA70OBlldyYhVW3ZSeONxbkMfeANHNxAKvu6vHHLsi+RJ6N+QwZuwKR9UZBQG8mjUFWm1kvpOQjSFeT51hQ1WVIlu1dppfm3gAqLtA3zrFmPQNpXnNI4NcAjsya+KQbDZHcdxaqNY3JRIFZ16Bm1RX2IgLwp35G+VBGNVe+liBndq/EFncy/FhVQZw6JRmTpt6Mpgirgo7Siq2k29cnzICPJWSklCuuDCGbv7HVxuHakUS/9V+KMMah0hc7lggzMhy1DAz7aywZeKwLBVzR58nsfp3BNykKCDq/42clDq3icV8+Io2fKR9yO9zJC9gZK5/0Kr6vT0+wdQSwECFAAUAAAACACvkFldpae9/zEAAAAwAAAADwAAAAAAAAAAAAAAtoEAAAAAc3JjL19faW5pdF9fLnB5UEsBAhQAFAAAAAgAsJBZXZ80gCR6AQAA/QIAAA0AAAAAAAAAAAAAALaBXgAAAHNyYy9hdG9taWMucHlQSwECFAAUAAAACAB7aF5dfuFdagUCAABSBAAAEQAAAAAAAAAAAAAAtoEDAgAAc3JjL2JvdW5kYXJpZXMucHlQSwECFAAUAAAACACxkFldmv+swIcFAABKDwAAEgAAAAAAAAAAAAAAtoE3BAAAc3JjL2NoZWNrcG9pbnRzLnB5UEsBAhQAFAAAAAgAGLdZXd3kQU6gAwAAowgAAA4AAAAAAAAAAAAAALaB7gkAAHNyYy9kYXRhc2V0LnB5UEsBAhQAFAAAAAgAo2heXfZfrlb5AwAAYwoAABAAAAAAAAAAAAAAALaBug0AAHNyYy9saXRlY3JhY2sucHlQSwECFAAUAAAACAB9aF5dcmfGQPMAAADDAQAAFAAAAAAAAAAAAAAAtoHhEQAAc3JjL21vZGVsX2ZhY3RvcnkucHlQSwECFAAUAAAACADyulldZkBMbEMEAADQDAAADQAAAAAAAAAAAAAAtoEGEwAAc3JjL21vZGVscy5weVBLAQIUABQAAAAIAPO6WV1RnfRq7QEAAMQEAAATAAAAAAAAAAAAAAC2gXQXAABzcmMvc2VnbWVudGF0aW9uLnB5UEsBAhQAFAAAAAgAd7loXZfoHZf1DgAAKiwAABkAAAAAAAAAAAAAALaBkhkAAHNjcmlwdHMvdHJhaW5fYmFzZWxpbmUucHlQSwECFAAUAAAACAB3uWhdW4TbZ0YLAACUGwAAGQAAAAAAAAAAAAAAtoG+KAAAc2NyaXB0cy9wcmVwYXJlX21pbGFuby5weVBLAQIUABQAAAAIAA+7aF3WDUfmKwEAAP4BAAAdAAAAAAAAAAAAAACAATs0AABjb25maWdzL21pbGFub19saXRlY3JhY2suanNvblBLAQIUABQAAAAIAA+7aF1UnbHrOgEAABsCAAAgAAAAAAAAAAAAAACAAaE1AABjb25maWdzL21pbGFub19saXRlY3JhY2tfYmEuanNvblBLBQYAAAAADQANAFYDAAAZNwAAAAA=')
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        target=ROOT/item.filename
        if target.exists() and target.read_bytes()!=z.read(item):
            raise ValueError('Code changed; use a fresh project/runtime')
        target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(z.read(item))
os.chdir(ROOT); sys.path.insert(0,str(ROOT))

import urllib.request, time
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024**2),b''): h.update(block)
    return h.hexdigest()
restore=Path('/kaggle/working')/'Milano_Resume_step13410.zip'
assert file_sha(restore)=='fbcb867fd6ac94b9b6aba3257d1f907bf64000fa2d427c94727fb20e9c0b51b0', 'Restoration package checksum mismatch'
with zipfile.ZipFile(restore) as z:
    assert z.testzip() is None, 'Restoration package corrupt'
    for name in z.namelist():
        target=(ROOT/name).resolve()
        assert target.is_relative_to(ROOT.resolve()), 'Unsafe restore path'
        if target.exists():
            assert target.read_bytes()==z.read(name), 'Restore conflicts with existing state'
    z.extractall(ROOT)
DATA=ROOT/'data'; manifest_path=DATA/'manifest.json'
identity=json.loads((ROOT/'restoration_identity.json').read_text())
assert file_sha(manifest_path)==identity['manifest_sha256']
manifest=json.loads(manifest_path.read_text())
assert manifest['status']=='FROZEN_MILANO_V1'
# Fetch original publisher bytes, while retaining exact normalized masks from the backup pack.
source=Path('/kaggle/working/milano_publisher_source.zip')
if not source.exists():
    partial=source.with_suffix('.zip.partial')
    url='https://zenodo.org/records/20040434/files/Pavement_Crack_Segmentation_Dataset.zip?download=1'
    for attempt in range(3):
        try:
            size=0
            with urllib.request.urlopen(url,timeout=45) as response, partial.open('wb') as f:
                for block in iter(lambda:response.read(4*1024**2),b''):
                    f.write(block); size+=len(block)
                    if size%(64*1024**2)==0: print('Publisher download bytes:',size,flush=True)
            partial.replace(source); break
        except Exception as exc:
            print('Publisher download retry:',type(exc).__name__,attempt+1,flush=True)
            if attempt==2: raise
            time.sleep(5)
h=hashlib.md5()
with source.open('rb') as f:
    for block in iter(lambda:f.read(1024**2),b''): h.update(block)
assert h.hexdigest()=='4b530eba05708a195355a0064e520db4', 'Publisher archive checksum mismatch'
with zipfile.ZipFile(source) as z:
    for row in manifest['records']:
        target=(DATA/row['image']).resolve()
        assert target.is_relative_to(DATA.resolve())
        if not target.exists():
            target.parent.mkdir(parents=True,exist_ok=True)
            with z.open(row['image']) as incoming, target.open('wb') as outgoing:
                shutil.copyfileobj(incoming,outgoing)
from src.dataset import verify_files
verify_files(manifest['records'],DATA)
print('Restored exact data and checkpoint state:',len(manifest['records']),'pairs',flush=True)

RECOVERY=None  # e.g. Path('/kaggle/input/my-backup/milano_recovery_litecrack_....zip')
if RECOVERY is not None:
    RECOVERY=Path(RECOVERY)
    assert file_sha(RECOVERY)==RECOVERY.with_suffix('.zip.sha256').read_text().split()[0]
    assert not (ROOT/'artifacts').exists(), 'Restore only into a fresh project'
    with zipfile.ZipFile(RECOVERY) as z:
        assert all((ROOT/n).resolve().is_relative_to(ROOT.resolve()) for n in z.namelist())
        z.extractall(ROOT)
runtime={'torch':torch.__version__,'torchvision':torchvision.__version__,
         'numpy':numpy.__version__,'pillow':PIL.__version__,'cuda':torch.version.cuda,
         'gpu':torch.cuda.get_device_name(),'python':sys.version}
lock=ROOT/'artifacts/runtime.json'
if lock.exists():
    expected=json.loads(lock.read_text())
    if expected!=runtime:
        # Rebuilt interpreter must retain the exact Python patch and all locked packages.
        assert {k:v for k,v in expected.items() if k!='python'}=={k:v for k,v in runtime.items() if k!='python'}, 'Recovery package or GPU runtime changed'
        assert expected['python'].split()[0]==runtime['python'].split()[0], 'Python patch version changed'
        evidence={'original_runtime':expected,'rebuilt_runtime':runtime,
                  'scope':'Exact locked versions restored; Python build provenance differs; original runtime.json retained. Cross-build bitwise equivalence not asserted.'}
        evidence_path=lock.parent/'runtime_rebuild.json'
        if evidence_path.exists(): assert json.loads(evidence_path.read_text())==evidence, 'Rebuilt runtime changed again'
        else: evidence_path.write_text(json.dumps(evidence,indent=2))
        print('Recorded rebuilt Python provenance; pinned versions match',flush=True)
else:
    lock.parent.mkdir(exist_ok=True); lock.write_text(json.dumps(runtime,indent=2))

(ROOT/'tests').mkdir(exist_ok=True)
(ROOT/'tests/test_phase2.py').write_text("import hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport pytest\n\ntorch=pytest.importorskip('torch')\nnp=pytest.importorskip('numpy')\nImage=pytest.importorskip('PIL.Image')\nfrom src.models import SegmentationUNet\nfrom src.segmentation import image_metrics,region_loss\nfrom scripts.train_baseline import train\n\n\ndef test_models_preserve_spatial_dimensions():\n    torch.set_num_threads(1)\n    for architecture in ('unet','r2au'):\n        model=SegmentationUNet(architecture,2)\n        assert model(torch.zeros(2,3,35,37)).shape==(2,1,35,37)\n\n\ndef test_empty_and_disjoint_metric_conventions():\n    logits=torch.full((1,1,16,16),-100.)\n    truth=torch.zeros_like(logits)\n    assert image_metrics(logits,truth)['dice'].item()==1\n    truth[:,:,2,2]=1\n    assert image_metrics(logits,truth)['dice'].item()==0\n    assert image_metrics(logits,truth)['iou'].item()==0\n\n\ndef test_bce_dice_recovers_gradient_for_saturated_wrong_prediction():\n    logits=torch.full((1,1,16,16),-100.,requires_grad=True)\n    truth=torch.ones_like(logits)\n    region_loss(logits,truth,'bce_dice').backward()\n    assert torch.isfinite(logits.grad).all() and (logits.grad<0).all()\n\n\ndef fixture_manifest(root):\n    rng=np.random.default_rng(123)\n    records=[]\n    splits=['source_train']*4+['source_validation']*2+['source_holdout']*2+['target_query']*2\n    for i,split in enumerate(splits):\n        image=rng.integers(0,256,(32,32,3),dtype=np.uint8)\n        mask=np.zeros((32,32),dtype=np.uint8); mask[:,i+2:i+4]=255\n        ip=root/f'{i}.png'; mp=root/f'{i}_mask.png'\n        Image.fromarray(image).save(ip); Image.fromarray(mask).save(mp)\n        records.append({'image_id':str(i),'image':ip.name,'mask':mp.name,'split':split,\n                        'image_sha256':hashlib.sha256(ip.read_bytes()).hexdigest(),\n                        'mask_sha256':hashlib.sha256(mp.read_bytes()).hexdigest()})\n    path=root/'manifest.json'\n    path.write_text(json.dumps({'status':'FROZEN_PILOT','records':records}))\n    return path\n\n\n@pytest.mark.parametrize('architecture',['unet','r2au','litecrack','litecrack_ba'])\n@pytest.mark.parametrize('device',['cpu','cuda'])\ndef test_actual_trainer_mid_epoch_resume(tmp_path,architecture,device):\n    if architecture.startswith('litecrack'): pytest.importorskip('torchvision')\n    if device=='cuda' and not torch.cuda.is_available(): pytest.skip('CUDA unavailable')\n    torch.set_num_threads(1)\n    manifest=fixture_manifest(tmp_path)\n    config={'architecture':architecture,'base_channels':2,'normalization':'group','seed':42,'image_size':32,\n            'batch_size':2,'epochs':2,'learning_rate':.001,'patience':10,\n            'checkpoint_every_steps':1,'device':device,'loss':'bce_dice','gradient_clip_norm':1.,'deterministic':True}\n    uninterrupted=tmp_path/'full'; recovered=tmp_path/'resumed'\n    train(config,manifest,tmp_path,uninterrupted,final_evaluation=False)\n    train(config,manifest,tmp_path,recovered,stop_after_steps=1,final_evaluation=False)\n    assert json.loads((recovered/'status.json').read_text())['state']=='stopped_safely'\n    train(config,manifest,tmp_path,recovered,resume=True,final_evaluation=False)\n    def latest(run):\n        pointer=json.loads((run/'checkpoints/latest.json').read_text())\n        return torch.load(run/'checkpoints'/pointer['file'],map_location='cpu',weights_only=False)\n    a,b=latest(uninterrupted),latest(recovered)\n    assert a['global_step']==b['global_step']==4\n    assert a['sampler_state']==b['sampler_state']\n    assert a['scheduler']==b['scheduler']\n    for key in a['model']: assert torch.equal(a['model'][key],b['model'][key]),key\n    assert a['optimizer']['param_groups']==b['optimizer']['param_groups']\n    for key in a['optimizer']['state']:\n        for field in a['optimizer']['state'][key]:\n            assert torch.equal(a['optimizer']['state'][key][field],b['optimizer']['state'][key][field])\n    changed={**config,'learning_rate':.1}\n    with pytest.raises(ValueError,match='Configuration changed'):\n        train(changed,manifest,tmp_path,recovered,resume=True,final_evaluation=False)\n")

test=subprocess.run([sys.executable,'-m','pytest','tests/test_phase2.py','-q',
    '--tb=short','-p','no:cacheprovider','--basetemp',str(ROOT/'gpu_recovery_test_tmp')])
assert test.returncode==0, 'GPU recovery tests failed; full training blocked'
(ROOT/'artifacts/gpu_tests.json').write_text(json.dumps({'state':'passed',
    'command':'pytest tests/test_phase2.py; CPU and CUDA model recovery tests'}))
from datetime import datetime, timezone
def export_recovery(arch):
    run=ROOT/'artifacts'/arch
    if not run.exists(): return
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    backup=ROOT.parent/f'milano_recovery_{arch}_{stamp}.zip'
    with zipfile.ZipFile(backup,'w',zipfile.ZIP_STORED) as z:
        # Preserve logs/config and all pointers, but only latest/best checkpoint payloads.
        for model_run in sorted((ROOT/'artifacts').iterdir()):
            if model_run.is_file():
                z.write(model_run,model_run.relative_to(ROOT).as_posix()); continue
            keep=set()
            pointers={}
            for pointer in model_run.glob('checkpoints/*.json'):
                content=pointer.read_bytes(); info=json.loads(content); target=pointer.parent/info['file']
                assert target.resolve().parent==pointer.parent.resolve()
                assert file_sha(target)==info['sha256'], 'Corrupt recovery pointer'
                keep.add(target.resolve())
                pointers[pointer.resolve()]=content
            for p in model_run.rglob('*'):
                if p.resolve() in pointers:
                    z.writestr(p.relative_to(ROOT).as_posix(),pointers[p.resolve()])
                elif p.is_file() and (p.suffix!='.pt' or p.resolve() in keep):
                    z.write(p,p.relative_to(ROOT).as_posix())
    backup.with_suffix('.zip.sha256').write_text(file_sha(backup)+'  '+backup.name+chr(10))
    from IPython.display import display, FileLink
    display(FileLink(str(backup)),FileLink(str(backup.with_suffix('.zip.sha256'))))

def run_model(arch):
    run=ROOT/'artifacts'/arch
    if (run/'status.json').exists() and json.loads((run/'status.json').read_text())['state']=='training_complete' and (run/'evaluation.json').exists():
        print('Already completed:',arch); export_recovery(arch); return
    assert shutil.disk_usage(ROOT).free>3*1024**3, 'Export and restore in fresh storage'
    cmd=[sys.executable,'scripts/train_baseline.py','--config',f'configs/milano_{arch}.json',
         '--manifest',str(manifest_path),'--data-root',str(DATA),'--run-dir',str(run)]
    if (run/'checkpoints/latest.json').exists(): cmd.append('--resume')
    process=subprocess.Popen(cmd)
    try:
        code=process.wait()
    except KeyboardInterrupt:
        import signal
        process.send_signal(signal.SIGINT)
        code=process.wait()
    finally:
        if process.poll() is not None: export_recovery(arch)
    assert code==0, 'Training failed; inspect failure.json and preserve recovery'

import threading, time, traceback
backup_stop=threading.Event()
def periodic_backup():
    while not backup_stop.wait(60):
        try:
            for arch in ('litecrack','litecrack_ba'):
                if (ROOT/'artifacts'/arch/'checkpoints/latest.json').exists():
                    export_recovery(arch)
                    break
        except Exception as error:
            print('PERIODIC BACKUP FAILED:',type(error).__name__,str(error),flush=True)
threading.Thread(target=periodic_backup,daemon=True).start()
try:
    run_model('litecrack')
    
    run_model('litecrack_ba')
    
    comparison={}
    for arch in ('litecrack','litecrack_ba'):
        run=ROOT/'artifacts'/arch
        assert json.loads((run/'status.json').read_text())['state']=='training_complete'
        comparison[arch]=json.loads((run/'evaluation.json').read_text())
    (ROOT/'artifacts/comparison.json').write_text(json.dumps(comparison,indent=2))
    print(json.dumps(comparison,indent=2))
    export_recovery('litecrack_ba')

finally:
    backup_stop.set()
